# Frequency Shift Keying Project - Part 2: FSK Transmitter

In [ ]:
## loa numpy and matplotlib
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

from IPython.display import Audio

## Background

A communications system aims to send information from one location to
another. In a digital communications system, the information consists
of a sequence of bits. Simply put, a digital communications system
uses the following steps to accomplish that goal:
* The transmitter is provided a sequence of bits that must be sent
  to the receiver; the receiver is in a different location from the
  transmitter.
* The transmitter maps the sequence of bits into an analog signal
  or waveform;
* The signal is sent to the receiver; this may happen via a cabled
  connection or wirelessly. As the signal travels from transmitter to
  the receiver it becomes weaker and it is disturbed by *noise*.
* The receiver observes this signal and employs knowledge of how
  the transmitter maps bits to signals to reconstruct the original bit
  sequence. 

In Part 1, you explored how textual information can be converted into a
sequence of bits and, also, how that sequence of bits can be returned
into textual format. This lab focuses on the second step and explores a concrete and
practical method for mapping a sequence of bits into an analog signal
that can be sent to the receiver.

### Import Code from Part 1

This part of the project builds on the functions that you developed in Part 1. 

To make these functions accessible, we need to store them in a *module*. To do so, proceed as follows.

1. In the directory that contains this notebook, create a file named `fsk.py`.
2. Near the top of your file place code to import NumPy:
``` Python
import numpy as np
```
3. Copy and paste the code for the four functions from part 1:
  * `int_to_bits()`
  * `bits_to_int()`
  * `string_to_bits()`
  * `bits_to_string()`

  To import your code and to verify that the imports worked, execute the following cell: 

In [ ]:
# import the top-level functions
from fsk import string_to_bits, bits_to_string

# test
str = "Information"
assert(str == bits_to_string(string_to_bits(str)))
print("OK")

## Frequency Shift Keying (FSK)

The digital modulation method used in this lab is called Frequency
Shift Keying (FSK).

In FSK, the sequence of bits to be sent is mapped into a sequence of tones.
As in all digital modulation methods, time is
defined into consecutive periods $T$ refered to as *bit periods*.

During each bit period, a single bit is transmitted. That is, the first
bit ($n=0$) is sent between times 0 and $T$ and the $n$-th bit is transmitted
from time $nT$ to $(n+1)T$.

Finally, if the $n$-th bit is $0$, then a tone of frequency $f_0$ is
sent. If the $n$-th bit is $1$, then a tone of frequency $f_1$ is
sent.

### Example

For example, assume that the bit sequence $[0,1, 1, 0]$ is to be
transmitted, that the bit period $T$ is 10 ms and the frequencies
$f_0$ and $f_1$ are 200 Hz and 400 Hz, respectively.
For these parameters, the resulting FSK signal is shown in the figure below.

Note that the frequency differs depending on whether a 0-bit or a 1-bit is sent.

![FSK Signal](sig.png)</br>
*An FSK signal for four bits $[0, 1, 1, 0]$ with $f_0=200$ Hz, $f_1=400$ Hz, and $T=10$ ms.*

## FSK Transmitter

Your assignment for this lab is to implement an FSK modulator in
NumPy and to integrate this modulator with the functions from part 1
to obtain a complete transmitter for sending textual information.


### Task: Implement a FSK Modulator

Write a Python/NumPy function `fsk_modulator` that implements a FSK modulator. 

Inputs to the function
are

* an array containing a sequence of bits
* the two FSK frequencies $f_0$ and $f_1$ (in Hertz)
* the bit period $T$ (in seconds)
* the sample rate $f_s$ (in Hertz)

<p style="margin-bottom: 15px; padding: 4px 12px; background-color: #e7f3fe; border-left: 6px solid #2196F3;">
<b>Hint:</b> You should probably pre-allocate the array that will hold the samples.
</p>

Thus, your function should start like this:


In [ ]:
def fsk_modulator(bits, f0, f1, T, fs):
    """FSK modulate a sequence of bits
    
    Parameters:
    ------------
    bits - the sequence of bits to be sent (must all be 0 or 1)
    f0 - frequency for 0-bits
    f1 - frequency for 1-bits
    T - bit period
    fs - sample rate

    Returns:
    --------
    Numpy array representing sequence of samples
    """
    N = int(T * fs)
    sig = np.zeros(len(bits) * N)
    tt=np.arange(0, T * len(bits), 1/fs)

    for n in range(len(bits)):
        if bits[n] == 0:
            f=f0
        else:
            f=f1
        sig[n*N:N*(n+1)]= np.cos(2*np.pi*f*tt[n*N:(n+1)*N])
    return sig 
    

To validate your function, recreate the plot above. 

* Generate a signal using your function with the following inputs:
`sig = fsk_modulator([0,1,1,0], 200, 400, 10e-3, 10e3)`.
* Create an appropriate time grid `tt`.
* Make a "nice" plot with a grid and properly labeled axes.

In [ ]:
# Plot
sig = fsk_modulator([0,1,1,0], 200, 400, 10e-3, 10e3)
tt=np.arange(0, 10e-3 * 4, 1/10e3)
plt.plot(tt, sig)

plt.grid()
plt.xlabel('Time (s)')

plt.show()

### Task: Transmitter for sending text

Combine the function `string_to_bits` with the FSK modulator
function `fsk_modulator` to form a complete transmitter. Name that function `fsk_transmitter`

This transmitter takes a text string as its input, converts  that
string to a bit sequence, and finally
modulates the bit sequence to produce an FSK signal.

Hence, your function should start like this:

In [ ]:
def fsk_transmitter(string, f0, f1, T, fs):
    """Modulate a string into an FSK signal
    Parameters:
    ------------
    string - the text to be transmitted
    f0 - frequency for 0-bits
    f1 - frequency for 1-bits
    T - bit period
    fs - sample rate

    Returns:
    --------
    Numpy array representing sequence of samples
    
    """
    bits = string_to_bits(string)

    res = fsk_modulator(bits, f0, f1, T, fs)

    return res
    

Invoke your function with the following parameters:

* set `string` to 'ECE 201'
* set the frequencies $f_0$ and $f_1$ to 4 and 7 KHz, respectively.
* set the bit duration $T$ to 20 ms
* set the sampling rate $f_s$ equal to 22.05 KHz

After synthesizing the FSK signal, play it using the Audio widget.

In [ ]:
# make the signal using fsk_transmitter
fs = 22.05e3
sig = fsk_transmitter('ECE 201', 4000, 7000, 20e-3, 22.05e3)


# play it through the speaker
Audio(sig, rate=fs)

### Spectrogram

Plot the spectrogram of the synthesized FSK signal; make sure that the axes are labeled accurately.

In your report, describe clearly how the spectrogram shows the
transmitted information and the parameters ($f_0$, $f_1$, and $T$) of the signal. 

Describe (qualitatively), a plausible way how
you can recover the bits (and then the string) from the spectrogram.

In [ ]:
# spectrogram
plt.specgram(sig, NFFT=256, Fs=fs)
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")

plt.show()

We could look at the frequencies that corresponds to 0 and 1 for each bit period and construct our sequence like that and we would then use our bits_to_string function to recover our text in this 7000 would correspong to 1s and 4000 to 0.

## Questions

In addition to performing and documenting the programming exercises above, in your
report also answer the following questions:
1. Assuming that the bit period $T$ is 2ms, how many bits per
  second can you transmitted? How many characters (assume that 8 bits are needed
  to represent a character)? Show your work.
2. Clearly, it is possible to speed up transmission by reducing the
  bit period $T$. Can you think of a way to transmit more bits per
  second without changing $T$? Be as specific as you can and quantify
  how much faster you can transmit.
3. Consider the example plot of a signal over time given at the start of this notebook. 
  How are the products
  $f_0T$ and $f_1T$ reflected in that plot? That is, compute the values
  for $f_0T$ and $f_1T$ for the given parameter values
  and determine how those two  values
  appear in the plot.

### Answers:

Use this cell and additional cells as needed to answer the questions above

1/.002 = 500 and 500 / 8 = 62.5 so we can send 62.5 characters per sec
2. we could add more frequencies representing a larger sequence of bits like freq 0 is 00 and freq 1 is 01 which would be twice as fast.
3. 200*.01 = 2 and 400*.01 = 4 they appear in the plot by how many cycles occur during that bit period 

## Summary

This was Part 2 of the Frequency Shift Keying (FSK) project. In this part of the project, we implemented part of an FSK transmitter, which generates an FSK signal from an array of bits. We also plotted the spectrogram of an FSK signal.

### Deliverable

Submit a PDF version of this notebook. Make sure that:
* All cells are properly typeset and the "Incomplete" cell near the top is removed.
* All code cells are complete (no `FILL_ME_IN` left).
* All functions are properly documented.
* All plots have proper axis labels and other adornments as appropriate.
* Answers are provided for the section entitled "Questions."